In [ ]:
import glob
import numpy as np
import pandas as pd
from scipy import signal
from functools import reduce
from bluemath_tk.distributions.nonstat_gev import NonStatGEV

In [ ]:
def extract_number(filename):
    return int(''.join(filter(str.isdigit, filename)))

station_files = glob.glob('/home/aapolola/prjs0911/GTSM_ERA5_Extension/sksurge_parquets/*.parquet')
station_files.sort(key=extract_number)
print(station_files)

In [ ]:
def run_regression(X, y):
    X_const = sm.add_constant(X)
    model = sm.OLS(y, X_const).fit()
    
    dw_test = durbin_watson(model.resid)
    omnibus_stat, omnibus_p = omni_normtest(model.resid)
    
    return model, X_const, dw_test, omnibus_p, y

In [ ]:
def check_normality(omnibus_p):
    if omnibus_p < 0.05:
        print("Residuals not normal (Omnibus).")
        return False  # Residuals are not normal
    else:
        print("Residuals appear normal (Omnibus).")
        return True  # Residuals are normal

In [ ]:
def summarize_model(model, X_const):
    
    param_names = X_const.columns
    
    params = pd.Series(model.params, index=param_names)
    std_errors = pd.Series(model.bse, index=param_names)
    p_values = pd.Series(model.pvalues, index=param_names)

    summary = {
        'intercept': params['const'] if 'const' in param_names else np.nan,
        'r_squared': model.rsquared,
        'adj_r_squared': model.rsquared_adj, 
        'f_statistic': getattr(model, 'fvalue', np.nan),
        'f_pvalue': getattr(model, 'f_pvalue', np.nan),
        'model_summary': model.summary()
    }

    for name in params.index:
        if name != 'const':
            summary[f'slope_{name}'] = params[name]
            summary[f'p_value_{name}'] = p_values[name]
            summary[f'CI_{name}'] = std_errors[name] * 1.96
    return summary

In [ ]:
one_station = station_files[3915] 
data = pd.read_parquet(one_station)
data

In [ ]:
data = data [['surge_time','skew_surge']]
data

In [ ]:
data = data[~((data['surge_time'] >= '1950-01-01') & 
            (data['surge_time'] <= '1950-01-04'))]
data = data[~((data['surge_time'] >= '2021-03-09') & 
                    (data['surge_time'] < '2021-03-10'))]
data = data.reset_index(drop=True)

data['detrended_surge'] = signal.detrend(data['skew_surge'])
data

In [ ]:
data['Year'] = data['surge_time'].dt.year
data['Month'] = data['surge_time'].dt.month
data['Month_name'] = data['surge_time'].dt.month_name().str[:3]
data.set_index("surge_time", inplace=True)
data

In [ ]:
Annual_quant  = data['detrended_surge'].resample('YE').max().reset_index()
Annual_quant['Year'] = Annual_quant['surge_time'].dt.year
print(f"Annual quantiles:\n{Annual_quant}")

In [ ]:
data_monthly_maxima_idxs = data.resample("YE")["detrended_surge"].idxmax()
data_monthly_maxima_idxs

In [ ]:
data_monthly_maxima = data.loc[data_monthly_maxima_idxs]
data_monthly_maxima

In [ ]:
climate_files = sorted(glob.glob('/home/aapolola/prjs0911/GTSM_ERA5_Extension/Climate_indices/*.txt'))
print (climate_files)

In [ ]:
pd.set_option('display.expand_frame_repr', False) 

In [ ]:
indices =['AMM', 'AO', 'Nao', 'Nino1+2_anom', 'Nino3_4_anom', 'Nino3_anom', 'Nino4_anom', 'ONI', 'PNA', 'SOI', 'WHWP', 'WP', 'PDO']
name_indices = ['Atlantic Multidecadal Oscilliation', 'Arctic Oscilliation', 'North Atlantic Oscilliation', 'Extreme Eastern Tropical Pacific SST',
            'East Central Tropical Pacific SST', 'Eastern Tropical Pacific SST', 'Central Tropical Pacific SST',  'Oceanic Nino Index',
            'Pacific North American Index', 'Southern Oscilliation Index', 'Western Hemisphere Warm Pool', 'Western Pacific Index', 'Pacific decadal oscillation']
dfs = {}
for mode, file in zip(indices, climate_files):
    df = pd.read_fwf(file, skiprows=1, header=None)
    df.columns = ["Year", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    df['mean'] = df.loc[:, 'Jan':'Dec'].mean(axis=1)
    dfs[mode] = df[['Year','mean']] 
print(dfs)

In [ ]:
dfs.items()

In [ ]:
melted_dfs = []
for mode, df in dfs.items():
    df_long = df.melt(id_vars='Year', var_name='Mean', value_name=mode)  # value_name = mode name
    melted_dfs.append(df_long)
melted_dfs

In [ ]:
merged_indices = reduce(lambda left, right: pd.merge(left, right, on=['Year', 'Mean']), melted_dfs)
merged_indices = merged_indices.drop(columns=['Mean'])
merged_indices

In [ ]:
data.set_index("surge_time", inplace=True)
data.index = pd.to_datetime(data.index)
data["year"] = data.index.year
data["month"] = data.index.month
data["day"] = data.index.day
data["hour"] = data.index.hour
data["minute"] = data.index.minute
data["second"] = data.index.second
data

In [ ]:
data_monthly_maxima_idxs = data.resample("M")["detrended_surge"].idxmax()
data_monthly_maxima_idxs
data_monthly_maxima = data.loc[data_monthly_maxima_idxs]
data_monthly_maxima

In [ ]:
'''
data_monthly_maxima = data_monthly_maxima.reset_index()
data_monthly_maxima
'''

In [ ]:
'''
dupli = data_monthly_maxima.duplicated(subset=["surge_time"])
print(f"Number of duplicate entries: {dupli.sum()}")
dat = data_monthly_maxima[dupli]
dat
'''

In [ ]:
data_monthly_maxima.detrended_surge.plot(figsize=(14, 7))

In [ ]:
days_in_month = {
    1: 31,
    2: 28.25,
    3: 31,
    4: 30,
    5: 31,
    6: 30,
    7: 31,
    8: 31,
    9: 30,
    10: 31,
    11: 30,
    12: 31,
}
days_in_month

In [ ]:
data_monthly_maxima

In [ ]:
data_monthly_maxima["time"] = (
    data_monthly_maxima["year"]
    - np.min(data_monthly_maxima["year"])
    + (data_monthly_maxima["month"] - 1) / 12
    + data_monthly_maxima["day"] / data_monthly_maxima["month"].map(days_in_month) / 12
)
data_monthly_maxima

In [ ]:

data_monthly_maxima_idxs = data_monthly_maxima.resample("Y")["skew_surge"].idxmax()
data_monthly_maxima_idxs
data_annual_maxima = data_monthly_maxima.loc[data_monthly_maxima_idxs]
data_annual_maxima


In [ ]:
data_annual_maxima.plot(x="year", y="skew_surge", figsize=(14, 7))

In [ ]:
data_monthly_maxima = data_monthly_maxima.reset_index()
data_monthly_maxima

In [ ]:
climate_files = '/home/aapolola/prjs0911/GTSM_ERA5_Extension/Climate_indices/NaoJones.txt'
df = pd.read_fwf(climate_files, skiprows=1, header=None)
df

In [ ]:
df.columns = ["Year", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
df['centile99.5'] = df.loc[:, 'Jan':'Dec'].mean(axis=1)
nf = df[['Year','centile99.5']] 
nf

In [ ]:
nf.set_index("Year", inplace=True)
nf

In [ ]:
nonstatgev = NonStatGEV(
    xt=data_annual_maxima["detrended_surge"].values,
    t=data_annual_maxima["time"].values,
    covariates=nf,
    harms=False,
    trends=False,
    var_name="skew_surge",
)
nonstatgev

In [ ]:
nonstatgev.fit(
    nmu=0,          # Harmonics in location
    npsi=0,         # Harmonics in scale
    ngamma=0,       # Harmonics in shape
    ntrend_loc=0,   # Trend in location
    list_loc="all", # List of covariates in location (if "all" all covariates are used)
    ntrend_sc=0,    # Trend in scale
    list_sc="all",  # List of covariates in scale (if "all" all covariates are used)
    ntrend_sh=0,    # Trend in shape
    list_sh="all"   # List of covariates in shape (if "all" all covariates are used)
)

In [ ]:
nonstatgev.plot()


In [ ]:
nonstatgev.ngamma

In [ ]:
nonstatgev.summary()